# LMO half-cell rate model: walkthrough

This notebook follows the seven steps of the plan in `CLAUDE.md` using the functions in `src/lmo/`. Run `python scripts/run_all.py` first (or `make all`) so that `results/*.json` and `data/processed/` exist; the slow steps (fits) are loaded from those files rather than re-run here.

In [ ]:
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from lmo import RATES, CVS, REPRESENTATIVE_CYCLE, RESULTS_DIR, PROCESSED_DIR
from lmo.io import load_gcd, get_step, gcd_summary, cv_cycle, cv_peaks
from lmo.plotting import COLORS
%matplotlib inline

## 1. Parser
Every GCD file is split into cycles and charge / discharge steps; capacity is the trapezoid integral of the measured current (the C/10 run reads +0.147 / -0.140 mA on the 10 mA range).

In [ ]:
tab = gcd_summary()
tab[tab.representative & (tab.step == 'discharge')][['rate','cycle','capacity_mAh','mean_voltage_V','energy_mWh','mean_current_mA','coulombic_efficiency']]

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
for rate in RATES:
    s = get_step(rate, REPRESENTATIVE_CYCLE[rate], 'discharge')
    ax.plot(s.capacity_mAh, s.voltage_V, color=COLORS[rate], label=rate)
ax.set_xlabel('Discharge capacity (mAh)'); ax.set_ylabel('Voltage vs Li/Li$^+$ (V)'); ax.legend();

## 2. Pseudo-OCV and dQ/dV
Average of the C/10 cycle-2 charge and discharge on a normalised-capacity axis, mapped to x in Li$_x$Mn$_2$O$_4$ with x = 1 at the 3.5 V end.

In [ ]:
ocv = pd.read_csv(PROCESSED_DIR / 'pseudo_ocv.csv')
meta = json.load(open(RESULTS_DIR / 'ocv.json'))
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(ocv.stoichiometry, ocv.charge_V, alpha=.5, label='charge'); ax[0].plot(ocv.stoichiometry, ocv.discharge_V, alpha=.5, label='discharge'); ax[0].plot(ocv.stoichiometry, ocv.ocv_V, 'k', label='pseudo-OCV')
ax[0].set_xlabel('x in Li$_x$Mn$_2$O$_4$'); ax[0].set_ylabel('V'); ax[0].legend()
for k in ('discharge','charge'):
    ic = pd.read_csv(PROCESSED_DIR / f'dqdv_c10_{k}.csv'); ax[1].plot(ic.voltage_V, ic.dqdv_mAh_V, label=k)
ax[1].set_xlim(3.8, 4.4); ax[1].set_xlabel('V'); ax[1].set_ylabel('dQ/dV (mAh/V)'); ax[1].legend()
meta['dqdv_peaks_discharge'], meta['stoichiometry_window']

## 3. Model
PyBaMM DFN / SPMe in half-cell mode (`working electrode = positive`) from the `Xu2019` lithium-metal set with the positive electrode replaced. The disc diameter is unknown; for 14 mm the capacity fixes the active volume fraction and porosity.

In [ ]:
from lmo.model import disc_geometry, simulate_gcd
pd.DataFrame([disc_geometry(d) for d in (12, 14, 16)]).set_index('disc_mm')

In [ ]:
# one forward simulation with the fitted SPMe parameters
fit = {n: v['value'] for n, v in json.load(open(RESULTS_DIR / 'fit_spme.json'))['parameters'].items() if n in ('Ds','k','R_contact')}
sol, frames = simulate_gcd(0.717, 3.85, fit=fit, kind='SPMe', rate='C/2')
m = get_step('C/2', 1, 'discharge')
plt.plot(m.capacity_mAh, m.voltage_V, color=COLORS['C/2'], label='measured'); plt.plot(frames[1].capacity_mAh, frames[1].voltage_V, 'k--', label='SPMe'); plt.legend(); plt.xlabel('mAh'); plt.ylabel('V');

## 4. Fit
Three effective parameters fitted to C/5 cycle 1 and C/2 cycle 1 (charge and discharge): $D_s$ (reported as $\tau = R^2/D_s$), the rate constant $k$ and a contact resistance. Jacobian 95% intervals and the spread of leave-one-step-out refits are both reported.

In [ ]:
for kind in ('spme', 'dfn'):
    p = (RESULTS_DIR / f'fit_{kind}.json')
    if p.exists():
        f = json.load(open(p)); print(kind.upper(), 'training RMSE %.1f mV' % (1000*f['training_rmse_V']))
        display(pd.DataFrame(f['parameters']).T)

## 5. Blind validation
2C cycle 2 was never used in fitting. The CVs are simulated by driving the model with the measured voltage programme.

In [ ]:
for kind in ('spme', 'dfn'):
    p = RESULTS_DIR / f'validation_{kind}.json'
    if p.exists():
        v = json.load(open(p)); print(kind.upper(), json.dumps(v['headline'], indent=1))

In [ ]:
from IPython.display import Image, display
display(Image('../figures/headline_measured_vs_model.png', width=900))
display(Image('../figures/cv_dfn.png', width=1000)) if (RESULTS_DIR / 'cv_dfn.json').exists() else display(Image('../figures/cv_spme.png', width=1000))

## 6. Diagnostics
Overpotential breakdown from PyBaMM's voltage-component variables and the CV b-value analysis.

In [ ]:
kind = 'dfn' if (RESULTS_DIR / 'overpotentials_dfn.json').exists() else 'spme'
ov = json.load(open(RESULTS_DIR / f'overpotentials_{kind}.json'))
pd.DataFrame({s['rate']: s['capacity_weighted_mean_mV'] for s in ov['steps'] if s['step']=='discharge'}).round(1)

In [ ]:
display(Image(f'../figures/overpotentials_{kind}.png', width=1000))
cv = json.load(open(RESULTS_DIR / f'cv_{kind}.json')); cv['b_value'], cv['randles_sevcik']['apparent_D_geometric_mean_m2_s'], cv['randles_sevcik']['fitted_Ds_m2_s']

## 7. Write-up
See `README.md` and `results/resume_bullets.md`; every number there is read from `results/*.json`.